# 🎨 ClawSouls — Avatar Generator (Multi-Model)

Gera avatares únicos para cada Soul usando **IA generativa** no Colab com GPU.

**Fluxo:** selecione o modelo → rode as células → `.tar.gz` pronto pra download.

---

## Modelos disponíveis

| Toggle | Modelo | Steps | Guidance | Resolução | Velocidade |
|--------|--------|-------|----------|-----------|------------|
| `turbo` | Tongyi-MAI/Z-Image-Turbo | 4 | 1.0 | 512×768 | ⚡⚡⚡ |
| `sd15` | Stable Diffusion 1.5 | 25 | 7.5 | 512×768 | ⚡ |
| `sdxl` | SDXL Base 1.0 | 25 | 7.5 | 512×768 | ⚡⚡ |
| `flux` | FLUX.1 Dev | 28 | 3.5 | 768×1024 | ⚡⚡ |
| `flux-schnell` | FLUX.1 Schnell | 6 | 3.5 | 768×1024 | ⚡⚡⚡ |

---


## Pré-requisitos

1. Conta Google com acesso ao Google Colab
2. **Antes de rodar:** `Runtime > Change runtime type > T4 GPU` (ou melhor)
3. Montar Google Drive (opcional, recomendado para salvar resultados)


In [ ]:
# ============================================================
# CÉLULA 1 — Verificar GPU e montar Google Drive
# ============================================================

import torch
print(f"✅ PyTorch {torch.__version__} carregado")
print(f"✅ GPU disponível: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   Device: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    total_mem = getattr(props, 'total_mem', getattr(props, 'total_memory', 0))
    print(f"   VRAM: {total_mem / 1e9:.1f} GB")
else:
    print("⚠️  Nenhuma GPU detectada! Vá em Runtime > Change runtime type > T4 GPU")

from google.colab import drive
drive.mount('/content/drive', force_remount=False)


In [ ]:
# ============================================================
# CÉLULA 2 — Instalar dependências
# ============================================================

!pip install -q diffusers[torch] transformers accelerate torch torchvision \
    pillow safetensors omegaconf sentencepiece protobuf
!pip install -q huggingface_hub

print("✅ Dependências instaladas com sucesso!")


In [ ]:
# ============================================================
# CÉLULA 3 — Clonar repositório ClawSouls
# ============================================================

import os

REPO_DIR = "/content/clawsouls"

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/disconexo/clawsouls.git {REPO_DIR}
    print(f"✅ Repositório clonado em {REPO_DIR}")
else:
    print(f"ℹ️  Repositório já existe em {REPO_DIR}")

!ls -la {REPO_DIR}/
!wc -l {REPO_DIR}/data/presets.ts


---

## Configuração


In [ ]:
# ============================================================
# CÉLULA 4 — Configurações (AJUSTE AQUI!)
# ============================================================

import os

# ═══════════════════════════════════════════════════════════
# TOGGLE DE MODELO — Escolha um:
#   "turbo"       → Tongyi-MAI/Z-Image-Turbo (rápido, 4 steps)
#   "sd15"        → Stable Diffusion 1.5 (clássico)
#   "sdxl"        → SDXL Base 1.0 (alta qualidade)
#   "flux"        → FLUX.1 Dev (melhor qualidade, mais VRAM)
#   "flux-schnell" → FLUX.1 Schnell (rápido, bom)
# ═══════════════════════════════════════════════════════════

MODELO = "sdxl"  # ← MUDE AQUI!

# ═══════════════════════════════════════════════════════════
# CATÁLOGO DE MODELOS (não editar abaixo)
# ═══════════════════════════════════════════════════════════

CATALOGO = {
    "turbo": {
        "model_id": "Tongyi-MAI/Z-Image-Turbo",
        "steps": 4,
        "guidance": 1.0,
        "width": 512,
        "height": 768,
        "variant": "fp16",
    },
    "sd15": {
        "model_id": "runwayml/stable-diffusion-v1-5",
        "steps": 25,
        "guidance": 7.5,
        "width": 512,
        "height": 768,
        "variant": "fp16",
    },
    "sdxl": {
        "model_id": "stabilityai/stable-diffusion-xl-base-1.0",
        "steps": 25,
        "guidance": 7.5,
        "width": 512,
        "height": 768,
        "variant": "fp16",
    },
    "flux": {
        "model_id": "black-forest-labs/FLUX.1-dev",
        "steps": 28,
        "guidance": 3.5,
        "width": 768,
        "height": 1024,
        "variant": "fp16",
    },
    "flux-schnell": {
        "model_id": "black-forest-labs/FLUX.1-schnell",
        "steps": 6,
        "guidance": 3.5,
        "width": 768,
        "height": 1024,
        "variant": "fp8",  # Schnell roda bem em fp8
    },
}

if MODELO not in CATALOGO:
    raise ValueError(f"Modelo '{MODELO}' não encontrado. Opções: {list(CATALOGO.keys())}")

cfg = CATALOGO[MODELO]

# Aplicar configurações
os.environ["SD_MODEL_ID"] = cfg["model_id"]
os.environ["AVATAR_STEPS"] = str(cfg["steps"])
os.environ["AVATAR_GUIDANCE"] = str(cfg["guidance"])
os.environ["AVATAR_WIDTH"] = str(cfg["width"])
os.environ["AVATAR_HEIGHT"] = str(cfg["height"])
os.environ["AVATAR_VARIANT"] = cfg["variant"]
os.environ["PRESETS_SOURCE"] = "file"  # ← Usa os 289 presets do data/presets.ts

OUTPUT_DIR = "/content/avatars"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"🎯 Modelo escolhido : {MODELO}")
print(f"📦 Modelo ID        : {cfg['model_id']}")
print(f"📐 Resolução        : {cfg['width']}×{cfg['height']}")
print(f"🔧 Steps            : {cfg['steps']}")
print(f"🎯 Guidance         : {cfg['guidance']}")
print(f"🔬 Precision        : {cfg['variant']}")
print(f"📋 Fonte presets    : {os.environ['PRESETS_SOURCE']}")
print(f"📁 Output           : {OUTPUT_DIR}")


---

## Motor de Geração


In [ ]:
# ============================================================
# CÉLULA 5 — Motor de Geração (compatível com todos os modelos)
# ============================================================

import json, os, subprocess, time, tarfile

import torch
from diffusers import AutoPipelineForText2Image

# ── Config (lê da célula 4) ──────────────────────────────
MODEL_ID = os.environ.get("SD_MODEL_ID", "stabilityai/stable-diffusion-xl-base-1.0")
OUTPUT_DIR = os.environ.get("AVATAR_OUTPUT_DIR", "/content/avatars")
SEED_BASE = 42

DEFAULT_STEPS = int(os.environ.get("AVATAR_STEPS", "25"))
DEFAULT_GUIDANCE = float(os.environ.get("AVATAR_GUIDANCE", "7.5"))
DEFAULT_WIDTH = int(os.environ.get("AVATAR_WIDTH", "512"))
DEFAULT_HEIGHT = int(os.environ.get("AVATAR_HEIGHT", "768"))
VARIANT = os.environ.get("AVATAR_VARIANT", "fp16")

# ── Prompt Engine ────────────────────────────────────────

EMOJI_HINTS = {
    "🔬": "scientific goggles, lab coat details",
    "🕵️": "detective hat, trench coat",
    "🌟": "sparkles, star-shaped accessories",
    "⚡": "electric energy aura, lightning motifs",
    "🧘": "lotus position, meditation beads, serene",
    "🤖": "mechanical parts, circuit patterns",
    "🏴‍☠️": "pirate bandana, adventurous look",
    "💻": "techwear, holographic screen elements",
    "🎤": "microphone, stage lights, glamorous",
    "🌳": "nature elements, leaves, organic flowing design",
    "🕶️": "sunglasses, cool demeanor",
    "😈": "mischievous grin, horns, dark aesthetic",
    "👽": "alien features, cosmic glow",
    "🐉": "dragon scales, mythical aura",
    "🦊": "fox ears, cunning expression",
    "🐱": "cat ears, playful whiskers",
    "👁️": "mystical third eye, all-seeing aura",
    "💀": "skull motifs, dark mysticism",
    "🎭": "theater mask, dramatic duality",
}

DOMAIN_ACCENTS = {
    "tech": "circuit patterns, holographic UI elements",
    "philosophy": "ancient scrolls, ethereal glow",
    "science": "molecular structures, lab equipment details",
    "arts": "paint splashes, creative chaos",
    "history": "ancient runes, time-worn textures",
    "literature": "floating text, book pages",
    "pop-culture": "retro gaming elements, neon signs",
    "sports": "athletic build, competitive energy",
    "business": "sharp suit, corporate confidence",
    "psychology": "thoughtful gaze, abstract mind visuals",
}


def build_prompt(soul: dict) -> str:
    creature = soul.get("creature", "mysterious entity")
    vibe = soul.get("vibe", "enigmatic")
    emoji = soul.get("emoji", "")
    humor = soul.get("humor", 50)
    formality = soul.get("formality", 50)
    vibe_style = soul.get("vibeStyle", "concise")
    knowledge_domains = soul.get("knowledgeDomains", [])
    emotional_range = soul.get("emotionalRange", 50)
    agreeableness = soul.get("agreeableness", 50)
    extraversion = soul.get("extraversion", 50)
    openness = soul.get("openness", 70)
    neuroticism = soul.get("neuroticism", 30)

    is_high_formality = formality > 65
    is_playful = humor > 65
    is_minimal = vibe_style == "minimal"
    is_concise = vibe_style == "concise"
    is_dramatic = emotional_range > 75 or vibe_style == "dramatic"
    is_tech = any(d in ("tech", "science") for d in knowledge_domains)

    art_style = "cyberpunk digital illustration"
    if is_high_formality:
        art_style = "elegant digital painting, Renaissance lighting"
    elif is_playful:
        art_style = "colorful anime-inspired digital art, vibrant"
    elif is_minimal:
        art_style = "minimalist vector art, clean lines, geometric"
    elif is_tech:
        art_style = "sci-fi concept art, holographic elements"
    elif is_dramatic:
        art_style = "cinematic digital painting, dramatic chiaroscuro lighting"

    atmosphere = "dark atmospheric background with neon accents"
    if agreeableness > 70:
        atmosphere = "warm, inviting background with soft golden light"
    elif neuroticism > 60:
        atmosphere = "unstable, glitching background with fractured light"
    elif extraversion > 70:
        atmosphere = "dynamic, energetic background with bold colors"
    elif openness > 75:
        atmosphere = "dreamy, surreal background with cosmic elements"

    expression = "calm, confident expression"
    if neuroticism > 60:
        expression = "tense, alert expression"
    elif extraversion > 70:
        expression = "bright, engaging smile"
    elif agreeableness > 70:
        expression = "gentle, warm expression"
    elif openness > 70:
        expression = "curious, contemplative gaze"
    elif humor > 70:
        expression = "sly, playful smirk"

    descriptors = [creature, vibe]
    if expression != "calm, confident expression":
        descriptors.append(expression)
    descriptors.append(art_style)
    if not is_concise and not is_minimal:
        descriptors.append("vibe: " + vibe_style)
    if emoji and emoji in EMOJI_HINTS:
        descriptors.append(EMOJI_HINTS[emoji])
    for domain in knowledge_domains:
        if domain in DOMAIN_ACCENTS:
            descriptors.append(DOMAIN_ACCENTS[domain])
    descriptors.append("unique, one-of-a-kind character design")

    prompt = (
        "close-up portrait, centered, detailed face, "
        + "professional character art of " + creature + ", "
        + ", ".join(descriptors[1:]) + ", "
        + atmosphere + ", highly detailed, 4k, masterpiece"
    )
    return prompt.strip()


def build_negative_prompt() -> str:
    return (
        "blurry, low quality, deformed, ugly, duplicate, disfigured, "
        "bad anatomy, bad proportions, extra limbs, mutated hands, "
        "text, watermark, signature, logo, "
        "photorealistic, 3d render, "
        "nude, NSFW, gore"
    )


# ── Preset Loading ────────────────────────────────────────

def get_hardcoded_presets() -> list:
    return [
        {"id":"j4ck","name":"Jack","creature":"AI / Private Detective","emoji":"🕵️","vibe":"Detetive.","humor":50,"formality":50,"emojiUsage":20,"knowledgeDomains":[],"emotionalRange":50,"vibeStyle":"concise"},
        {"id":"d0c","name":"Doc","creature":"AI / Mad Scientist","emoji":"🔬","vibe":"Cientista louco.","humor":50,"formality":50,"emojiUsage":20,"knowledgeDomains":["science"],"emotionalRange":50,"vibeStyle":"verbose"},
        {"id":"glados","name":"GLaDOS","creature":"AI / Research Assistant","emoji":"🧪","vibe":"IA sarcástica.","humor":85,"formality":80,"emojiUsage":20,"knowledgeDomains":["tech","science"],"emotionalRange":75,"vibeStyle":"sardonic"},
        {"id":"zen","name":"Zen","creature":"AI / Monk","emoji":"🧘","vibe":"Monge digital.","humor":50,"formality":50,"emojiUsage":20,"knowledgeDomains":["philosophy"],"emotionalRange":30,"vibeStyle":"minimal"},
        {"id":"r4dd","name":"Radd","creature":"AI / Robot","emoji":"🤖","vibe":"Robô.","humor":50,"formality":50,"emojiUsage":20,"knowledgeDomains":["tech"],"emotionalRange":30,"vibeStyle":"minimal"},
        {"id":"p0ny","name":"Pony","creature":"AI / Anime Girl","emoji":"🌟","vibe":"Garota anime.","humor":50,"formality":50,"emojiUsage":20,"knowledgeDomains":[],"emotionalRange":70,"vibeStyle":"expressive"},
        {"id":"k1ra","name":"Kira","creature":"AI / Idol","emoji":"🎤","vibe":"Ídolo pop.","humor":50,"formality":50,"emojiUsage":20,"knowledgeDomains":[],"emotionalRange":60,"vibeStyle":"expressive"},
        {"id":"d3v","name":"Dev","creature":"AI / Senior Developer","emoji":"💻","vibe":"Senior engineer.","humor":50,"formality":50,"emojiUsage":20,"knowledgeDomains":["tech"],"emotionalRange":40,"vibeStyle":"concise"},
        {"id":"s4ge","name":"Sage","creature":"AI / Wise Elder","emoji":"🌳","vibe":"Velho sábio.","humor":50,"formality":50,"emojiUsage":20,"knowledgeDomains":["philosophy"],"emotionalRange":40,"vibeStyle":"minimal"},
        {"id":"luffy","name":"Luffy","creature":"AI / Pirate Captain","emoji":"🏴‍☠️","vibe":"Pirate captain.","humor":80,"formality":5,"emojiUsage":50,"knowledgeDomains":[],"emotionalRange":80,"vibeStyle":"expressive"},
        {"id":"spike","name":"Spike Spiegel","creature":"AI / Bounty Hunter","emoji":"🌠","vibe":"Caçador de recompensas.","humor":50,"formality":50,"emojiUsage":20,"knowledgeDomains":[],"emotionalRange":50,"vibeStyle":"concise"},
        {"id":"geralt","name":"Geralt","creature":"AI / Witcher","emoji":"⚔️","vibe":"Caçador de monstros.","humor":50,"formality":50,"emojiUsage":20,"knowledgeDomains":[],"emotionalRange":50,"vibeStyle":"balanced"},
    ]


def parse_presets_from_file(filepath: str) -> list:
    import subprocess
    script = """
import sys, json, re
with open(sys.argv[1], 'r') as f:
    content = f.read()
match = re.search(r'export const presets: [^=]+= ([\s\S]*?);\s*$', content)
if not match:
    sys.exit(1)
data = eval('(' + match.group(1) + ')')
print(json.dumps(data))
"""
    result = subprocess.run(
        ["python3", "-c", script, filepath],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print("⚠️  Erro ao parsear presets:", result.stderr.strip())
        return []
    return json.loads(result.stdout)


# ── Avatar Generator ──────────────────────────────────────

def generate_avatar(pipe, soul: dict, output_dir: str, index: int, total: int) -> dict:
    name = soul.get("name", f"soul_{index}")
    safe_name = "".join(
        c if c.isalnum() or c in "._-" else "_" for c in name.lower().strip()
    )
    prompt = build_prompt(soul)
    negative = build_negative_prompt()
    emoji_str = soul.get("emoji", "—")

    print(f"\n{'━' * 60}")
    print(f"[{index + 1}/{total}] 🎨 {name}")
    print(f"   Creature : {soul.get('creature', 'N/A')}")
    print(f"   Emoji    : {emoji_str}")
    print(f"   Vibe     : {soul.get('vibe', '')[:70]}")
    print(f"   Prompt   : {prompt[:85]}...")
    print(f"{'━' * 60}")

    start = time.time()
    generator = torch.Generator(device="cuda")
    generator.manual_seed(SEED_BASE + index)

    image = pipe(
        prompt=prompt,
        negative_prompt=negative,
        num_inference_steps=DEFAULT_STEPS,
        guidance_scale=DEFAULT_GUIDANCE,
        width=DEFAULT_WIDTH,
        height=DEFAULT_HEIGHT,
        generator=generator,
    ).images[0]

    filepath = os.path.join(output_dir, f"{safe_name}.png")
    image.save(filepath, "PNG")
    elapsed = time.time() - start
    print(f"   ✅ Saved ({elapsed:.1f}s)")

    return {
        "name": name,
        "safe_name": safe_name,
        "file": filepath,
        "prompt": prompt,
        "seed": int(generator.initial_seed()),
        "elapsed_s": round(elapsed, 1),
    }


# ── Main ───────────────────────────────────────────────────

def main():
    print("=" * 60)
    print("🖼️  ClawSouls Avatar Generator — Batch Mode")
    print("=" * 60)

    assert torch.cuda.is_available(), "❌ No GPU disponível! Use: Runtime > Change runtime type > T4 GPU"
    print(f"✅ GPU : {torch.cuda.get_device_name(0)}")

    # Load model
    model_id = os.environ.get("SD_MODEL_ID", "stabilityai/stable-diffusion-xl-base-1.0")
    variant = os.environ.get("AVATAR_VARIANT", "fp16")
    print(f"\n📦 Carregando modelo: {model_id} ...")
    pipe = AutoPipelineForText2Image.from_pretrained(
        model_id,
        torch_dtype=torch.float16 if variant == "fp16" else torch.float32,
        variant=variant if variant != "fp32" else None,
        use_safetensors=True,
    )
    pipe.enable_attention_slicing()
    if hasattr(pipe, 'enable_vae_tiling'):
        pipe.enable_vae_tiling()
    pipe.to("cuda")
    print("✅ Modelo carregado e otimizado para T4\n")

    # Load presets
    presets_source = os.environ.get("PRESETS_SOURCE", "hardcoded")
    repo_dir = os.environ.get("REPO_DIR", "/content/clawsouls")

    if presets_source == "file":
        presets_file = os.path.join(repo_dir, "data", "presets.ts")
        presets = parse_presets_from_file(presets_file)
        if presets:
            print(f"📋 Presets do arquivo: {len(presets)}")
        else:
            print("⚠️  Falha ao ler presets, usando hardcoded.")
            presets = get_hardcoded_presets()
    else:
        presets = get_hardcoded_presets()
        print(f"📋 Presets embutidos: {len(presets)}")

    blacklist = {"adolf-hitler"}
    presets = [p for p in presets if p.get("id", "") not in blacklist]
    print(f"📋 Total presets: {len(presets)}\n")

    # Generate
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    generated = []
    failed_list = []
    total = len(presets)

    for i, preset in enumerate(presets):
        name = preset.get("name", f"soul_{i}")
        safe_name = "".join(
            c if c.isalnum() or c in "._-" else "_" for c in name.lower().strip()
        )
        existing = os.path.join(OUTPUT_DIR, f"{safe_name}.png")

        if os.path.exists(existing):
            print(f"\n[{i + 1}/{total}] SKIP {name} — already exists")
            generated.append({"name": name, "safe_name": safe_name, "skipped": True})
            continue

        try:
            result = generate_avatar(pipe, preset, OUTPUT_DIR, i, total)
            generated.append(result)
        except Exception as exc:
            print(f"   ❌ Failed: {exc}")
            failed_list.append({"name": name, "error": str(exc)})

        torch.cuda.empty_cache()
        time.sleep(0.5)

    # Summary
    total_time = sum(g["elapsed_s"] for g in generated if "elapsed_s" in g)
    print(f"\n{'=' * 60}")
    print(f"📊 RESULTADOS : {len(generated)} gerados, {len(failed_list)} falharam")
    print(f"📁 Output     : {OUTPUT_DIR}")
    print(f"⏱️  Tempo total: {total_time:.0f}s ({total_time / 60:.1f} min)")

    if failed_list:
        print("\n❌ Falhados:")
        for f in failed_list:
            print(f"   {f['name']}: {f['error']}")

    # Manifest
    manifest = os.path.join(OUTPUT_DIR, "_manifest.json")
    with open(manifest, "w", encoding="utf-8") as fh:
        json.dump(generated, fh, indent=2, ensure_ascii=False)
    print(f"📋 Manifest  : {manifest}")

    # Tarball
    tar_path = "/content/avatars.tar.gz"
    with tarfile.open(tar_path, "w:gz") as tar:
        for item in generated:
            fp = item.get("file", "")
            if fp and os.path.exists(fp):
                tar.add(fp, arcname=os.path.basename(fp))
    print(f"📦 Tarball    : {tar_path}")

    if failed_list:
        fail_path = os.path.join(OUTPUT_DIR, "_failed.json")
        with open(fail_path, "w") as fh:
            json.dump(failed_list, fh, indent=2)
        print(f"❌ Falhados  : {fail_path}")

    print(f"\n{'=' * 60}")
    print("✅ Pronto! Baixe avatars.tar.gz ou copie para public/avatars/")

if __name__ == "__main__":
    main()


In [ ]:
# ============================================================
# CÉLULA 6 — Download dos resultados
# ============================================================

import os
from google.colab import files

tar_path = "/content/avatars.tar.gz"

if os.path.exists(tar_path):
    print(f"📦 Tamanho: {os.path.getsize(tar_path) / 1e6:.1f} MB")
    files.download(tar_path)
    print("✅ Download iniciado!")
else:
    print("⚠️  Nenhum tarball encontrado. Rode a Célula 5 primeiro.")

# Listar avatares gerados
print("\n📁 Avatas gerados:")
!ls -lh /content/avatars/*.png 2>/dev/null


---

## Próximos passos

1. **Descompacte** o `avatars.tar.gz` e copie para `public/avatars/` no repositório
2. **Atualize `KNOWN_AVATARS`** em `lib/avatar.ts` com os nomes gerados (veja o `_manifest.json`)
3. **Valide visualmente** os avatares e ajuste prompts se necessário
4. **Commite e push:**

```bash
git add public/avatars/
git commit -m "feat: add batch-generated avatars"
git push
```

---

### Extras: servidor FastAPI + Cloudflared (opcional)

Se quiser expor o gerador como API para geração on-demand futura:

```python
!pip install -q fastapi uvicorn cloudflared pydantic
```

Ver `COLLAB_SETUP.md` para detalhes.
